<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 16px; text-align: center; margin-bottom: 20px;">
  <h1 style="color: #FF7A00; font-size: 2.4em; margin-bottom: 8px;">✈️ International Airline Passengers</h1>
  <h2 style="color: #ffffff; font-size: 1.3em; font-weight: 300;">Time Series Analysis & Forecasting</h2>
  <p style="color: #aaaaaa; margin-top: 16px;">A Portfolio Project by Meroe Asgharbeigi · Aarhus, Denmark</p>
  <p style="color: #aaaaaa;">Tools: Python · Pandas · Statsmodels · SARIMA · Matplotlib · Seaborn</p>
</div>

---

## 🗺️ Project Overview

This project performs a complete **time series analysis** on monthly international airline passenger counts from **1949 to 1960**. The dataset is a classic benchmark in time series analysis, originally published by Box & Jenkins (1976).

**The business scenario:**  
> *It is September 1961. An investment firm has asked you to analyse the past decade of airline passenger data and advise whether the aviation industry is a sound investment — and what to expect in the coming year.*

**We answer this by working through:**
1. Cleaning and preparing the data
2. Exploring patterns through statistics and visualisation
3. Decomposing the series into trend, seasonality, and residual
4. Testing for stationarity and transforming the series
5. Reading ACF/PACF plots to select a forecasting model
6. Fitting a **SARIMA** model and evaluating its performance
7. Delivering a written business interpretation

---

## 📋 Table of Contents
| # | Section |
|---|---|
| 1 | Importing Libraries |
| 2 | Loading the Dataset |
| 3 | Analysing the Data |
| 4 | Cleaning the Data |
| 5 | Exploratory Data Analysis (EDA) |
| 6 | Visualisation |
| 7 | Decomposition |
| 8 | Stationarity Testing |
| 9 | Making the Series Stationary |
| 10 | ACF & PACF Analysis |
| 11 | SARIMA Forecasting Model |
| 12 | Forecast Evaluation |
| 13 | Business Interpretation & Conclusions |

---
## 1. 📦 Importing Libraries

Before we do anything, we load the tools we need. Think of libraries as specialised toolboxes — each one gives us ready-made functions so we don't have to write everything from scratch.

| Library | What it does |
|---|---|
| `pandas` | Load, clean, and manipulate tabular data |
| `numpy` | Fast numerical calculations |
| `matplotlib` / `seaborn` | Create visualisations |
| `statsmodels` | Statistical tests and time series models |
| `sklearn` | Calculate model accuracy metrics |

In [ ]:
# ── Standard libraries ──────────────────────────────────────────────────────
import datetime          # Built-in Python: working with dates
import warnings
warnings.filterwarnings('ignore')   # Keep output clean

# ── Data wrangling ──────────────────────────────────────────────────────────
import pandas as pd      # DataFrames — the backbone of data analysis
import numpy as np       # Numerical arrays and math operations

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Time series statistics ───────────────────────────────────────────────────
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ── Model evaluation ─────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Global plot style ────────────────────────────────────────────────────────
# Sets a consistent, clean look for all our charts
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

ORANGE = '#FF7A00'   # Brand colour for highlights
BLUE   = '#1f77b4'
RED    = '#d62728'
GREEN  = '#2ca02c'

print('✅ All libraries imported successfully.')

---
## 2. 📂 Loading the Dataset

> **❓ Question:** What does our raw data look like, and did it load correctly?

We read the CSV file with `pd.read_csv()`. The first thing we always do after loading is inspect the data — checking column names, data types, and the first few rows. It is easy to spot problems here (wrong delimiter, extra whitespace, garbled column names) before they cause silent errors downstream.

In [ ]:
# Load the CSV — the file sits in the same folder as this notebook
df_raw = pd.read_csv('AirPassengers.csv')

# Always start by looking at the shape and the first few rows
print(f'Shape: {df_raw.shape}  →  {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
print(f'Columns: {df_raw.columns.tolist()}')
print()
df_raw.head(10)

> **💡 Answer:** The dataset loaded correctly. We have 144 rows (12 years × 12 months) and 2 columns: `Month` (stored as a string like `'1949-01'`) and `#Passengers` (an integer). The `#` symbol and capital letter in the column name will cause problems in code — we fix this in the cleaning step.

---
## 3. 🔍 Analysing the Data

> **❓ Question:** Do we have missing values, correct data types, and a reasonable range of values?

Before touching the data, we take a full inventory. This step tells us:
- Whether any rows are missing
- Whether columns have the right types (e.g. dates should not be strings)
- Whether the numbers make sense (no negative passengers, no impossible values)

In [ ]:
print('=== DATA TYPES ===')
# dtypes tells us what Python thinks each column is
print(df_raw.dtypes)

print('\n=== MISSING VALUES ===')
# isnull().sum() counts how many NaN / None values are in each column
print(df_raw.isnull().sum())

print('\n=== BASIC STATISTICS ===')
# describe() gives us count, mean, std, min, max and quartiles
df_raw.describe()

> **💡 Answer:** No missing values in the raw file. However, `Month` is an `object` (string) — not a proper date — and `#Passengers` has an unfriendly name. The passenger range (104 to 622) looks realistic for an era when air travel was growing rapidly. We now know exactly what needs to be fixed.

---
## 4. 🧹 Cleaning the Data

> **❓ Question:** How do we turn raw, messy data into an analysis-ready time series?

Good cleaning is invisible — it solves problems quietly so that every step after it is reliable. We will:
- **Rename** columns to clean, lowercase names (no `#`, no spaces)
- **Parse** the date string into a real `datetime` object
- **Extract** year and month as separate integer columns (useful for grouping later)
- **Interpolate** any missing values using neighbouring values (best practice: fill gaps, don't drop rows)
- **Add** a data-density column (observations per year) to confirm we have complete years
- **Set the datetime as the index** — this is what makes pandas treat it as a proper time series

In [ ]:
# ── Work on a copy so the raw data is always available for reference
df = df_raw.copy()

# ── Step 1: Rename columns — no '#', no capitals, no spaces
# '#Passengers' is a valid column name but a nightmare to type in code
df.rename(columns={'Month': 'month_str', '#Passengers': 'passengers'}, inplace=True)
print('Columns after rename:', df.columns.tolist())

# ── Step 2: Parse the date string → proper datetime
# '%Y-%m' means four-digit year, dash, two-digit month  →  e.g. '1949-01'
df['date']  = pd.to_datetime(df['month_str'], format='%Y-%m')
df['year']  = df['date'].dt.year      # e.g. 1949
df['month'] = df['date'].dt.month     # e.g. 1  (January)

# ── Step 3: Interpolate missing values
# Linear interpolation: fills a gap with the straight-line value between neighbours
# This is better than dropping rows (which would break the monthly regularity)
missing_before = df['passengers'].isnull().sum()
df['passengers'] = df['passengers'].interpolate(method='linear')
missing_after  = df['passengers'].isnull().sum()
print(f'Missing values  →  before: {missing_before}, after: {missing_after}')

# ── Step 4: Data-density column — how many months of data do we have per year?
# This is a quality check: we expect 12 for every year
df['obs_in_year'] = df.groupby('year')['passengers'].transform('count')

# ── Step 5: Set column order and drop the original string column
df = df[['date', 'year', 'month', 'passengers', 'obs_in_year']].copy()

# ── Step 6: Set the datetime as the index with monthly frequency
# 'MS' = Month Start — tells pandas this is a monthly time series
df.set_index('date', inplace=True)
df.index.freq = 'MS'

print(f'\nDate range: {df.index.min().date()} → {df.index.max().date()}')
print(f'Total observations: {len(df)}')
print(f'All years complete (12 months each): {(df["obs_in_year"] == 12).all()}')
print()
df.head(13)

In [ ]:
# Final check: confirm data types are correct after cleaning
print('Data types after cleaning:')
print(df.dtypes)

> **💡 Answer:** The data is now fully clean. The date index is a proper `DatetimeIndex`, we have clean integer columns for year and month, and confirmed there are exactly 12 observations per year. We are ready to analyse.

---
## 5. 📊 Exploratory Data Analysis (EDA)

> **❓ Question:** What are the basic patterns in this data — which years were busiest, which months peak, and is there a trend over time?

EDA is the detective work phase. We use statistics to form hypotheses before we look at a single chart. If our numbers and our visuals agree, we can be confident in our conclusions.

In [ ]:
# ── Overall descriptive statistics
# This tells us the full range our data moves in
print('=== DESCRIPTIVE STATISTICS — Monthly Passengers ===')
print(df['passengers'].describe().round(1))

In [ ]:
# ── Which year had the highest and lowest total passenger volume?
yearly_total = df.groupby('year')['passengers'].sum()

print('Total passengers per year:')
print(yearly_total.to_string())
print(f"\n📈 Best year  : {yearly_total.idxmax()} — {yearly_total.max():,} passengers")
print(f"📉 Worst year : {yearly_total.idxmin()} — {yearly_total.min():,} passengers")
print(f"\n📊 Growth 1949→1960: {((yearly_total[1960]/yearly_total[1949])-1)*100:.1f}% total increase")

In [ ]:
# ── Which month peaks each year?
# We group by year, then find the row index (date) where passengers is maximum,
# then extract the month number from that index.
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

peak_month_num = df.groupby('year')['passengers'].idxmax().dt.month
peak_value     = df.groupby('year')['passengers'].max()

peak_df = pd.DataFrame({
    'peak_month'      : peak_month_num.map(lambda m: MONTH_NAMES[m-1]),
    'peak_passengers' : peak_value
})
print('Month with the highest passenger count each year:')
print(peak_df.to_string())

In [ ]:
# ── Yearly mean — is the average rising over time?
# A rising mean is evidence of a positive trend
yearly_mean = df.groupby('year')['passengers'].mean().round(1)
print('Yearly mean of monthly passengers:')
print(yearly_mean.to_string())

In [ ]:
# ── Yearly standard deviation — is the spread growing too?
# A rising std alongside a rising mean is the signature of MULTIPLICATIVE seasonality
# (the swings get bigger as the baseline grows)
yearly_std = df.groupby('year')['passengers'].std().round(1)
print('Yearly standard deviation of monthly passengers:')
print(yearly_std.to_string())

In [ ]:
# ── Average passengers by calendar month — the seasonality fingerprint
# Averaged across all 12 years, which month is typically busiest?
monthly_avg = df.groupby('month')['passengers'].mean()

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(MONTH_NAMES, monthly_avg.values,
              color=sns.color_palette('YlOrRd', 12))
ax.set_title('Average Passengers by Calendar Month (1949–1960)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Average Passengers')
for bar, val in zip(bars, monthly_avg.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.0f}', ha='center', va='bottom', fontsize=8.5)
plt.tight_layout()
plt.savefig('monthly_avg_bar.png', bbox_inches='tight')
plt.show()

> **💡 Answer — what the numbers tell us:**
> - The yearly mean **rises every single year**, from 126 (1949) to 476 (1960) — a **strong, unbroken upward trend**.
> - The yearly standard deviation **also rises** in step with the mean. This is the key diagnostic sign of **multiplicative seasonality** — the seasonal swings grow in proportion to the underlying level.
> - The peak month is **July or August every single year without exception** — summer holiday travel is a dominant, reliable force.
> - The 12-year total growth is roughly **280%**, which is extraordinary and confirms this is an industry in rapid expansion.

---
## 6. 📈 Visualisation

> **❓ Question:** What does the time series actually look like, and can we see the trend and seasonality with our own eyes?

Numbers confirm patterns; charts make them *visible*. We build four progressively more detailed views of the same data.

In [ ]:
# ── Plot 1: The raw time series ──────────────────────────────────────────────
# This is the first thing any analyst should do — look at the data as a line over time.
# The shaded area under the line makes the growth visually obvious.

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['passengers'], color=BLUE, linewidth=1.5, label='Monthly Passengers')
ax.fill_between(df.index, df['passengers'], alpha=0.12, color=BLUE)
ax.set_title('International Airline Passengers — Monthly (1949–1960)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Number of Passengers (thousands)')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.savefig('raw_series.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: Rolling Mean & Standard Deviation ────────────────────────────────
# A 12-month rolling window smooths out the seasonal wiggles so we can see
# whether the MEAN and STD are moving over time.
# If the red line is not flat → trend present (non-stationary mean)
# If the green line is not flat → variance changes over time (non-constant variance)

rolling_mean = df['passengers'].rolling(window=12).mean()
rolling_std  = df['passengers'].rolling(window=12).std()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['passengers'], color=BLUE,  alpha=0.5, linewidth=1.2, label='Original')
ax.plot(rolling_mean.index, rolling_mean, color=RED,   linewidth=2.2, label='Rolling Mean (12m)')
ax.plot(rolling_std.index,  rolling_std,  color=GREEN, linewidth=2.2, linestyle='--', label='Rolling Std (12m)')
ax.set_title('Rolling Mean & Standard Deviation (window = 12 months)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Passengers')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.savefig('rolling_stats.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Year × Month Heatmap ─────────────────────────────────────────────
# A heatmap shows every Year-Month combination as a coloured cell.
# Darker colour = more passengers. This gives us the seasonal pattern
# AND the year-over-year growth in a single glance.

pivot = df.pivot_table(values='passengers', index='year', columns='month', aggfunc='sum')
pivot.columns = MONTH_NAMES

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.4,
            ax=ax, cbar_kws={'label': 'Passengers'})
ax.set_title('Passenger Heatmap — Year × Month', fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Year')
plt.tight_layout()
plt.savefig('heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 4: Monthly Boxplot ───────────────────────────────────────────────────
# A boxplot for each calendar month shows us the DISTRIBUTION, not just the average.
# The box covers the middle 50% of values across all years.
# Wide boxes in July/August mean the summer peak is also the most variable month.

df['month_name'] = df['month'].map(lambda m: MONTH_NAMES[m-1])

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df, x='month_name', y='passengers', order=MONTH_NAMES,
            palette='YlOrRd', ax=ax)
ax.set_title('Distribution of Monthly Passengers by Calendar Month (all years)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Passengers')
plt.tight_layout()
plt.savefig('boxplot_month.png', bbox_inches='tight')
plt.show()

> **💡 Answer — what the charts tell us:**
> - **Raw series**: A clear upward staircase with regular seasonal spikes. The peaks get visibly taller each year — this is the signature of multiplicative seasonality.
> - **Rolling stats**: Both the red (mean) and green (std) lines slope upward — confirming the series is **non-stationary** in both mean and variance.
> - **Heatmap**: The rightmost columns (summer months) are consistently dark, confirming July/August as the peak season every year. The rows get progressively darker from top to bottom — the growth trend is clear.
> - **Boxplot**: The July and August boxes sit highest and are widest, meaning they are both the busiest *and* the most variable months. November and January are the quietest and most predictable.
>
> **Key visual conclusion**: This series has a strong trend, strong seasonality, and the two are coupled — the series is definitively **non-stationary** and will require transformation before modelling.

---
## 7. 🔬 Decomposition

> **❓ Question:** If we separate the trend, seasonality, and noise — what does each component look like, and which decomposition model (additive vs multiplicative) is the right one?

**Decomposition** surgically separates a time series into three layers:
- **Trend** — the long-term direction (up, down, flat)
- **Seasonal** — the repeating calendar pattern (monthly, quarterly, etc.)
- **Residual** — what is left after removing the trend and seasonality (should look like random noise)

**Additive model**: `Series = Trend + Seasonal + Residual`  
→ Use when the seasonal swings stay the same size regardless of the trend level.

**Multiplicative model**: `Series = Trend × Seasonal × Residual`  
→ Use when the seasonal swings grow *proportionally* with the trend level.

Because our EDA showed that the standard deviation grows with the mean, **multiplicative is the correct choice** here.

In [ ]:
# Set up a clean date-indexed series for decomposition
# We need the index to be the date, with no extra columns
series_for_decomp = df['passengers'].copy()
series_for_decomp.index.freq = 'MS'

# ── Multiplicative decomposition with period = 12 (monthly data, annual season)
decomp = seasonal_decompose(series_for_decomp, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
components = [
    ('Observed (Original Data)', decomp.observed,  BLUE),
    ('Trend Component',          decomp.trend,     RED),
    ('Seasonal Component',       decomp.seasonal,  ORANGE),
    ('Residual Component',       decomp.resid,     'purple')
]
for ax, (label, data, color) in zip(axes, components):
    ax.plot(data, color=color, linewidth=1.4)
    ax.set_ylabel(label, fontsize=9.5)
    ax.grid(True, alpha=0.4)

axes[0].set_title('Multiplicative Decomposition: Observed = Trend × Seasonal × Residual',
                  fontsize=13, fontweight='bold')
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('decomposition.png', bbox_inches='tight')
plt.show()

> **💡 Answer — reading the decomposition:**
> - **Trend**: A smooth, almost perfectly linear upward curve. No recessions, no dips — purely consistent growth through the 1950s aviation boom.
> - **Seasonal**: The seasonal component oscillates around **1.0** (the neutral multiplier). In July, the seasonal factor is approximately **1.3** — meaning July passengers are 30% *above* the trend line. In November, the factor is around **0.75** — 25% *below*.
> - **Residual**: Fluctuates around **1.0** with no obvious pattern — mostly random. A well-behaved residual confirms that our two-component model (trend + seasonality) has captured the structure well.
>
> The multiplicative model was the correct choice: the seasonal component is expressed as a ratio (multiplier), not as a fixed offset.

---
## 8. 🧪 Stationarity Testing

> **❓ Question:** Is this time series stationary, and how do we prove it with a statistical test?

Most forecasting models (ARIMA, SARIMA) require the series to be **stationary** — meaning its statistical properties (mean, variance) do not change over time.

We use two complementary tests:

| Test | Null Hypothesis | Reject if |
|---|---|---|
| **ADF** (Augmented Dickey-Fuller) | Series has a unit root (non-stationary) | p-value < 0.05 |
| **KPSS** (Kwiatkowski–Phillips–Schmidt–Shin) | Series is stationary | p-value < 0.05 |

Using both tests together gives us a more robust conclusion than relying on ADF alone.

In [ ]:
def check_stationarity(series, name='Series'):
    """Run ADF + KPSS tests and plot rolling statistics.
    
    Parameters
    ----------
    series : pd.Series   — The time series to test
    name   : str         — Label for the plot title and printed output
    """
    print(f'\n{"="*58}')
    print(f'  Stationarity Tests  →  {name}')
    print(f'{"="*58}')

    # Rolling window statistics — visual stationarity check
    # A stationary series should show a flat mean and flat std
    rol_mean = series.rolling(12).mean()
    rol_std  = series.rolling(12).std()

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(series,   color=BLUE,  linewidth=1.2, label='Original')
    ax.plot(rol_mean, color=RED,   linewidth=2.0, label='Rolling Mean (12m)')
    ax.plot(rol_std,  color=GREEN, linewidth=2.0, linestyle='--', label='Rolling Std (12m)')
    ax.set_title(f'Rolling Statistics — {name}', fontsize=12, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # ── ADF Test ──────────────────────────────────────────────────────────
    # H0: series is non-stationary (has a unit root)
    # We want to REJECT H0, so we need p < 0.05
    adf_result = adfuller(series.dropna(), autolag='AIC')
    adf_stat, adf_p = adf_result[0], adf_result[1]
    adf_verdict = '✅ Stationary' if adf_p < 0.05 else '❌ Non-stationary'

    print(f'\n[ADF Test]  H₀: series is non-stationary')
    print(f'  Test Statistic : {adf_stat:.4f}')
    print(f'  p-value        : {adf_p:.4f}  →  {adf_verdict}')
    for k, v in adf_result[4].items():
        print(f'  Critical ({k})  : {v:.4f}')

    # ── KPSS Test ─────────────────────────────────────────────────────────
    # H0: series IS stationary
    # We want to FAIL to reject H0, so we need p > 0.05
    kpss_stat, kpss_p, _, _ = kpss(series.dropna(), regression='c', nlags='auto')
    kpss_verdict = '✅ Stationary' if kpss_p > 0.05 else '❌ Non-stationary'

    print(f'\n[KPSS Test]  H₀: series is stationary')
    print(f'  Test Statistic : {kpss_stat:.4f}')
    print(f'  p-value        : {kpss_p:.4f}  →  {kpss_verdict}')

    # ── Joint verdict ─────────────────────────────────────────────────────
    both_stationary = (adf_p < 0.05) and (kpss_p > 0.05)
    if both_stationary:
        verdict = '🟢 CONCLUSION: Both tests agree — STATIONARY'
    elif (adf_p >= 0.05) and (kpss_p <= 0.05):
        verdict = '🔴 CONCLUSION: Both tests agree — NON-STATIONARY'
    else:
        verdict = '🟡 CONCLUSION: Tests disagree — further investigation needed'
    print(f'\n  {verdict}\n')

In [ ]:
# Test the original raw series
check_stationarity(df['passengers'], 'Original Series (raw passenger counts)')

> **💡 Answer:** As expected from the visual inspection:
> - **ADF p-value > 0.05** → we cannot reject non-stationarity
> - **KPSS p-value < 0.05** → we reject the stationarity hypothesis
> - **Both tests agree: the series is non-stationary.**
>
> This means we cannot directly feed this data into a forecasting model. We need to transform it first.

---
## 9. 🔧 Making the Series Stationary

> **❓ Question:** How do we remove the trend and seasonality mathematically, and how do we know when we have gone far enough?

We apply three transformations in sequence, testing stationarity after each step:

| Step | Transformation | What it fixes |
|---|---|---|
| 1 | **Log transform** | Stabilises the growing variance (multiplicative → additive) |
| 2 | **Seasonal differencing (lag 12)** | Removes the annual seasonality |
| 3 | **First-order differencing (lag 1)** | Removes the remaining trend |

**Differencing** means subtracting each value from the value a certain number of steps back:  
`diff(lag=1): y[t] - y[t-1]` (removes trend)  
`diff(lag=12): y[t] - y[t-12]` (removes annual seasonality)

In [ ]:
# ── Step 1: Log transformation ───────────────────────────────────────────────
# log() compresses large values more than small values,
# which flattens the growing amplitude and stabilises variance.
# This converts multiplicative structure to additive structure.

df['log_passengers'] = np.log(df['passengers'])

check_stationarity(df['log_passengers'], 'Step 1: Log-Transformed Series')

In [ ]:
# ── Step 2: Seasonal differencing (lag = 12) ─────────────────────────────────
# We subtract from each month the value from the SAME MONTH one year ago.
# This removes the annual seasonal cycle.
# Example: Jan-1950 minus Jan-1949 = the year-over-year change for January

df['log_diff12'] = df['log_passengers'].diff(12)

check_stationarity(df['log_diff12'].dropna(), 'Step 2: Log + Seasonal Diff (lag=12)')

In [ ]:
# ── Step 3: First-order differencing on the seasonal-differenced series ───────
# We subtract from each value the value from the PREVIOUS MONTH.
# This removes the remaining upward trend that seasonal differencing alone left behind.
# The result should now be stationary — a flat mean and flat variance.

df['log_diff1_12'] = df['log_diff12'].diff(1)

check_stationarity(df['log_diff1_12'].dropna(), 'Step 3: Log + Seasonal Diff + First-Order Diff')

> **💡 Answer — the transformation journey:**
> - **After log**: Variance is stabilised, but the series still has a clear upward mean → still non-stationary.
> - **After seasonal diff (lag 12)**: Seasonality is largely removed. The series is flatter, but ADF p-value may still be borderline.
> - **After first-order diff (lag 1)**: Both ADF and KPSS now confirm stationarity. The rolling mean is flat and the rolling std is flat.
>
> **Why this matters for modelling**: In SARIMA notation, this means `d=1` (one first-order diff) and `D=1` (one seasonal diff) — these are directly the parameters we will pass to the model.

---
## 10. 📉 ACF & PACF Analysis

> **❓ Question:** How do we choose the right SARIMA parameters (p, q, P, Q) from the data itself?

The **Autocorrelation Function (ACF)** and **Partial Autocorrelation Function (PACF)** plots are the standard diagnostic tools for parameter selection:

| Plot | What it shows | Informs |
|---|---|---|
| **ACF** | Correlation of the series with its own past values at each lag | MA order (`q`) and seasonal MA order (`Q`) |
| **PACF** | Correlation *after* removing the effect of intermediate lags | AR order (`p`) and seasonal AR order (`P`) |

**Reading rule**:
- A spike that crosses the blue confidence band at **lag 1** in PACF → `p = 1`
- A spike at **lag 1** in ACF → `q = 1`
- A spike at **lag 12** in PACF → `P = 1`
- A spike at **lag 12** in ACF → `Q = 1`

In [ ]:
stationary_series = df['log_diff1_12'].dropna()

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# ── ACF plot ──────────────────────────────────────────────────────────────
# Blue shaded band = 95% confidence interval
# Spikes outside the band are statistically significant
plot_acf(stationary_series, lags=36, ax=axes[0], color=BLUE, alpha=0.05)
axes[0].set_title('ACF — Autocorrelation Function (stationary series)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Lag (months)')

# ── PACF plot ─────────────────────────────────────────────────────────────
plot_pacf(stationary_series, lags=36, ax=axes[1], color=RED, alpha=0.05, method='ywm')
axes[1].set_title('PACF — Partial Autocorrelation Function (stationary series)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Lag (months)')

# Add vertical reference lines at lag 12 to guide the eye
for ax in axes:
    ax.axvline(x=12, color=ORANGE, linestyle=':', linewidth=1.5, alpha=0.7, label='Lag 12')
    ax.legend()

plt.tight_layout()
plt.savefig('acf_pacf.png', bbox_inches='tight')
plt.show()

print("""
How to read these plots:
  • Each bar = the correlation between the series and itself at that many months ago
  • Blue shaded zone = 95% confidence band (spikes inside are just noise)
  • Significant spike at lag  1 in PACF → AR order p = 1
  • Significant spike at lag  1 in ACF  → MA order q = 1  
  • Significant spike at lag 12 in ACF  → Seasonal MA Q = 1
  • Significant spike at lag 12 in PACF → Seasonal AR P = 1
  → Model choice: SARIMA(1,1,1)(1,1,1)[12]
""")

> **💡 Answer — parameter selection:**
> - The PACF shows a significant spike at lag 1 and a cut-off → `p = 1`
> - The ACF shows a significant spike at lag 1 and a gradual decay → `q = 1`
> - Both plots show significant spikes at lag 12 → `P = 1`, `Q = 1`
> - From our differencing steps: `d = 1`, `D = 1`
>
> **Selected model: SARIMA(1, 1, 1)(1, 1, 1)[12]**

---
## 11. 🤖 SARIMA Forecasting Model

> **❓ Question:** How well can we predict the future from past patterns, and how do we build an honest evaluation?

**SARIMA** (Seasonal AutoRegressive Integrated Moving Average) is the standard model for seasonal time series. The full notation is:

```
SARIMA(p, d, q)(P, D, Q)[s]
       ─────────  ───────────
       Non-seasonal  Seasonal
       
p = AR order (how many past values to use)
d = differencing order (how many times to difference)
q = MA order (how many past errors to use)
P, D, Q = same but at the seasonal frequency
s = season length (12 = monthly data with annual seasonality)
```

We use a **train / test split** to evaluate honestly: the model sees only the training data, then we compare its predictions against the real test values it never saw.

In [ ]:
# ── Train/Test Split ─────────────────────────────────────────────────────────
# We train on 1949–1958 (120 months) and hold out 1959–1960 (24 months) for testing.
# 24 months = 2 full seasonal cycles → a meaningful test.
# We work on the LOG scale to match what we did in the differencing steps.

series_log = df['log_passengers']

train = series_log[:'1958-12']   # Everything up to and including Dec 1958
test  = series_log['1959-01':]   # Jan 1959 onward → 24 months

print(f'Train: {train.index[0].date()} → {train.index[-1].date()}  ({len(train)} months)')
print(f'Test : {test.index[0].date()}  → {test.index[-1].date()}  ({len(test)} months)')

In [ ]:
# ── Fit SARIMA(1,1,1)(1,1,1)[12] ─────────────────────────────────────────────
# enforce_stationarity=False and enforce_invertibility=False are recommended
# for SARIMA models — they allow the optimizer more freedom.

model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

results = model.fit(disp=False)   # disp=False suppresses the optimizer output
print(results.summary())

In [ ]:
# ── Model Diagnostics ─────────────────────────────────────────────────────────
# A well-fitted model should produce residuals that look like white noise:
# → Standardised residuals: no pattern, centred around 0
# → Histogram: approximately normal (bell-shaped)
# → Q-Q plot: points follow the diagonal line
# → Correlogram: no significant autocorrelation at any lag

fig = results.plot_diagnostics(figsize=(14, 8))
fig.suptitle('SARIMA(1,1,1)(1,1,1)[12] — Residual Diagnostics',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('sarima_diagnostics.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Generate Forecasts ────────────────────────────────────────────────────────
# We forecast enough steps to cover the test period (24m) plus 12 months into the future
n_forecast = len(test) + 12

forecast_obj  = results.get_forecast(steps=n_forecast)
forecast_mean = forecast_obj.predicted_mean          # On log scale
conf_int      = forecast_obj.conf_int(alpha=0.05)    # 95% confidence interval

# ── Back-transform from log scale to original passenger scale
# np.exp() is the inverse of np.log()
fc_passengers  = np.exp(forecast_mean)
ci_lower       = np.exp(conf_int.iloc[:, 0])
ci_upper       = np.exp(conf_int.iloc[:, 1])
train_original = np.exp(train)
test_original  = np.exp(test)

# ── Forecast Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(train_original, color=BLUE,   linewidth=1.5, label='Training Data (1949–1958)')
ax.plot(test_original,  color=ORANGE, linewidth=2.0, label='Actual (Test 1959–1960)')
ax.plot(fc_passengers[:len(test)],    color=RED,    linewidth=2.0,
        linestyle='--', label='SARIMA Forecast (Test Period)')
ax.plot(fc_passengers[len(test):],    color='purple', linewidth=2.0,
        linestyle='--', label='SARIMA Forecast (Future 12 months)')
ax.fill_between(fc_passengers.index, ci_lower, ci_upper,
                color=RED, alpha=0.10, label='95% Confidence Interval')
ax.axvline(x=test_original.index[0], color='gray', linestyle=':', linewidth=1.5)
ax.text(test_original.index[0], ax.get_ylim()[0] + 20, '  Train | Test',
        color='gray', fontsize=9)

ax.set_title('SARIMA(1,1,1)(1,1,1)[12] — Forecast vs Actual Passengers',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Passengers (thousands)')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('forecast.png', bbox_inches='tight')
plt.show()

---
## 12. 📏 Forecast Evaluation

> **❓ Question:** How accurate is our model, and can we trust it for business decisions?

We evaluate the model using three metrics — each captures a different aspect of accuracy:

| Metric | Formula | What it tells us |
|---|---|---|
| **MAE** | mean(\|actual − forecast\|) | Average error in the original units (passengers) |
| **RMSE** | √mean((actual − forecast)²) | Penalises large errors more — sensitive to outliers |
| **MAPE** | mean(\|actual − forecast\| / actual) × 100 | Percentage error — scale-independent, easy to communicate |

In [ ]:
# ── Compute metrics on the TEST SET ONLY ────────────────────────────────────
# We never evaluate on training data — that would be cheating
actual    = test_original.values
predicted = fc_passengers[:len(test)].values

mae  = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
mape = np.mean(np.abs((actual - predicted) / actual)) * 100

print('╔══════════════════════════════════════════════╗')
print('║         FORECAST EVALUATION — TEST SET       ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  MAE  (passengers)  :  {mae:>8.1f}              ║')
print(f'║  RMSE (passengers)  :  {rmse:>8.1f}              ║')
print(f'║  MAPE (%)           :  {mape:>8.2f}%             ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Model AIC          :  {results.aic:>8.2f}              ║')
print(f'║  Model BIC          :  {results.bic:>8.2f}              ║')
print('╚══════════════════════════════════════════════╝')

In [ ]:
# ── Residual Analysis ────────────────────────────────────────────────────────
# Good residuals should: look like random noise (no pattern), be centred at 0,
# and be approximately normally distributed.

residuals = pd.Series(actual - predicted, index=test_original.index)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residuals over time
axes[0].plot(residuals, color=BLUE, linewidth=1.2)
axes[0].axhline(0, color=RED, linestyle='--', linewidth=1.2)
axes[0].set_title('Residuals Over Time', fontweight='bold')
axes[0].set_ylabel('Actual − Forecast')

# Residual distribution
sns.histplot(residuals, kde=True, ax=axes[1], color=BLUE, bins=10)
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual Value')

# Actual vs Predicted scatter
axes[2].scatter(actual, predicted, color=BLUE, alpha=0.7, edgecolors='white', s=60)
mn, mx = min(actual.min(), predicted.min())*0.95, max(actual.max(), predicted.max())*1.05
axes[2].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect Forecast')
axes[2].set_xlabel('Actual Passengers'); axes[2].set_ylabel('Predicted Passengers')
axes[2].set_title('Actual vs Predicted', fontweight='bold')
axes[2].legend()

plt.suptitle('Forecast Residual Analysis — Test Set (1959–1960)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('residuals.png', bbox_inches='tight')
plt.show()

> **💡 Answer — model performance:**
> - A **MAPE below 5%** is considered excellent for a monthly forecast on a 24-month horizon.
> - The residuals fluctuate randomly around zero with no obvious pattern — the model has captured all the structure in the data.
> - The scatter plot points cluster tightly around the perfect-forecast diagonal — systematic bias is minimal.
> - **Conclusion**: The model is reliable enough to support business decisions, provided confidence intervals are communicated alongside point forecasts.

---
## 13. 💼 Business Interpretation & Conclusions

> **The scenario**: It is September 1961. An investment firm asks: *Should we invest in the aviation industry? What does the data say?*

---

### 📌 Finding 1 — The trend is unambiguously upward

International passenger numbers grew from 112,000 in January 1949 to 432,000 in December 1960 — a **~280% increase** in 12 years, or roughly **12–15% compound annual growth**. The rolling mean chart shows this growth is smooth and uninterrupted — there was no recession, no disruption, no slowdown visible in this data. For an investor, this is the most important signal: demand for air travel grew every single year throughout the 1950s.

**Recommendation**: The long-term trend strongly supports investment in the aviation sector.

---

### 📌 Finding 2 — Seasonality is predictable and exploitable

The seasonal decomposition confirms that July and August account for approximately **30% more passengers than the trend line** in any given year. November and January are about **25% below** the trend. This seasonal pattern is **extraordinarily stable** — it repeated identically every year from 1949 to 1960. It is driven by summer holiday demand and is essentially predictable years in advance.

**Business implications**:
- **Staffing and capacity**: Airlines should plan for a July–August surge of approximately 30% above their average monthly baseline — every year, without exception.
- **Pricing strategy**: The predictable off-peak (Nov–Jan) is an opportunity for promotional pricing to stimulate demand during otherwise quiet months.
- **The September question**: The firm asks about September 1961 specifically. The data shows that September is consistently approximately 15% below the July peak — but it is still above the annual average. The seasonal drop from August to September is normal, expected, and should not be interpreted as a downturn.

---

### 📌 Finding 3 — The SARIMA model validates this outlook

Our SARIMA(1,1,1)(1,1,1)[12] model, trained on 1949–1958 and tested on 1959–1960, achieved a MAPE below 5% on a 24-month unseen hold-out period. This means:
- The historical patterns are **genuinely predictive** — they are not noise
- The model's 12-month forward forecast for 1961 projects **continued growth** with a summer peak even larger than 1960
- The 95% confidence intervals are relatively narrow, reflecting high predictability

**In plain language for the investment firm**: *The airline industry in the 1950s was one of the most predictable growth stories in economic history. The data supports confident investment. The only risk would be a structural external shock (conflict, fuel crisis) that nothing in this historical data could predict.*

---

### 📌 Limitations & Next Steps

| Limitation | What could be done |
|---|---|
| No external variables | Add GDP, fuel prices, or major events as regressors (SARIMAX) |
| Single model | Compare against Holt-Winters (ETS), Facebook Prophet, and LSTM neural networks |
| CI widens beyond 12 months | For longer horizons, use scenario modelling instead of point forecasts |
| No structural break detection | Real-world data (e.g. COVID for modern aviation) requires regime-switching models |

---

### 📌 Skills Demonstrated in This Project

| Skill | Where applied |
|---|---|
| Data cleaning & datetime engineering | Section 4 |
| Descriptive statistics & EDA | Section 5 |
| Time series visualisation (line, heatmap, boxplot) | Section 6 |
| Decomposition & model type selection (additive vs multiplicative) | Section 7 |
| Statistical hypothesis testing (ADF + KPSS) | Section 8 |
| Variance stabilisation and differencing | Section 9 |
| ACF/PACF reading for parameter selection | Section 10 |
| SARIMA model fitting and diagnostics | Section 11 |
| MAE, RMSE, MAPE evaluation on held-out data | Section 12 |
| Business interpretation for a non-technical audience | Section 13 |